In [1]:
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    accuracy_score, precision_score, recall_score, f1_score,
    precision_recall_curve,
)
from sklearn.preprocessing import StandardScaler

FEATURE_DIR = r"E:/Python/MSc_Project_Upgrade/datasets/bdt"
BASE_DIR    = r"E:/Python/MSc_Project_Upgrade/results_analysis/bdt"

In [2]:
WORKING_POINTS = [0.30, 0.50, 0.70, 0.90]
PT_BINS        = [10, 20, 30, 40, 50, 60, 75, 90, 110, 130, 155]
TEST_ENERGIES  = [100, 125, 150, 200, 250, 300]
 
FEATURE_NAMES = [
    "pt", "mass", "ncharged", "nneutrals",
    "ehad", "chf", "nef",
    "tau1", "tau21", "tau32",
]
 
MODEL_CONFIGS = {
    "rf_125": {
        "type": "rf", "train_energy": 125,
        "results_dir": os.path.join(BASE_DIR, "RF_model1_125GeV"),
        "model_file":  "rf_model1_125GeV.pkl",
        "scaler_file": "rf_model1_scaler.npz",
        "label": "Random Forest (125 GeV)", "color": "royalblue",
    },
    "rf_250": {
        "type": "rf", "train_energy": 250,
        "results_dir": os.path.join(BASE_DIR, "RF_model2_250GeV"),
        "model_file":  "rf_model2_250GeV.pkl",
        "scaler_file": "rf_model2_scaler.npz",
        "label": "Random Forest (250 GeV)", "color": "steelblue",
    },
    "xgb_125": {
        "type": "xgb", "train_energy": 125,
        "results_dir": os.path.join(BASE_DIR, "XGB_model1_125GeV"),
        "model_file":  "xgb_model1_125GeV.json",
        "scaler_file": "xgb_model1_scaler.npz",
        "label": "XGBoost (125 GeV)", "color": "darkorange",
    },
    "xgb_250": {
        "type": "xgb", "train_energy": 250,
        "results_dir": os.path.join(BASE_DIR, "XGB_model2_250GeV"),
        "model_file":  "xgb_model2_250GeV.json",
        "scaler_file": "xgb_model2_scaler.npz",
        "label": "XGBoost (250 GeV)", "color": "tomato",
    },
    "lgbm_125": {
        "type": "lgbm", "train_energy": 125,
        "results_dir": os.path.join(BASE_DIR, "LGBM_model1_125GeV"),
        "model_file":  "lgbm_model1_125GeV.txt",
        "scaler_file": "lgbm_model1_scaler.npz",
        "label": "LightGBM (125 GeV)", "color": "forestgreen",
    },
    "lgbm_250": {
        "type": "lgbm", "train_energy": 250,
        "results_dir": os.path.join(BASE_DIR, "LGBM_model2_250GeV"),
        "model_file":  "lgbm_model2_250GeV.txt",
        "scaler_file": "lgbm_model2_scaler.npz",
        "label": "LightGBM (250 GeV)", "color": "mediumseagreen",
    },
}

In [3]:
def load_scaler(scaler_path):
    d = np.load(scaler_path)
    sc = StandardScaler()
    sc.mean_  = d["mean"].astype(np.float64)
    sc.scale_ = d["scale"].astype(np.float64)
    sc.var_   = sc.scale_ ** 2
    sc.n_features_in_ = len(sc.mean_)
    return sc
 
 
def load_model(cfg):
    model_path = os.path.join(cfg["results_dir"], cfg["model_file"])
    if cfg["type"] == "rf":
        return joblib.load(model_path)
    elif cfg["type"] == "xgb":
        m = xgb.XGBClassifier()
        m.load_model(model_path)
        return m
    elif cfg["type"] == "lgbm":
        booster = lgb.Booster(model_file=model_path)
        class LGBMWrapper:
            def __init__(self, b): self.b = b
            def predict_proba(self, X):
                p = self.b.predict(X)
                return np.column_stack([1 - p, p])
            @property
            def feature_importances_(self):
                imp = self.b.feature_importance(importance_type="gain")
                return imp / imp.sum()
        return LGBMWrapper(booster)
 
 
def optimal_threshold(y_true, y_score):
    """F1-maximising threshold via vectorised precision_recall_curve."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    denom = precision[:-1] + recall[:-1]
    f1s   = np.where(denom > 0, 2 * precision[:-1] * recall[:-1] / denom, 0.0)
    return float(thresholds[np.argmax(f1s)])
 
 
def bkg_rejection_at_wp(y_true, y_score, sig_eff):
    """1/FPR at the threshold where TPR ≈ sig_eff."""
    fpr, tpr, _ = roc_curve(y_true, y_score)
    idx = np.argmin(np.abs(tpr - sig_eff))
    return np.inf if fpr[idx] == 0 else 1.0 / fpr[idx]
 
 
def pt_binned_efficiency(y_true, y_score, jet_pt, threshold, pt_bins):
    """
    Signal tagging efficiency and background fake rate binned by jet pT.
    Returns: bin_centres, sig_eff_per_bin, bkg_fake_per_bin
    """
    preds = (y_score >= threshold).astype(int)
    centres, sig_effs, bkg_fakes = [], [], []
    for lo, hi in zip(pt_bins[:-1], pt_bins[1:]):
        m_pt  = (jet_pt >= lo) & (jet_pt < hi)
        m_sig = m_pt & (y_true == 1)
        m_bkg = m_pt & (y_true == 0)
        centres.append(0.5 * (lo + hi))
        sig_effs.append(preds[m_sig].mean() if m_sig.sum() > 0 else np.nan)
        bkg_fakes.append(preds[m_bkg].mean() if m_bkg.sum() > 0 else np.nan)
    return np.array(centres), np.array(sig_effs), np.array(bkg_fakes)
 
 
def per_event_tau_efficiency(y_true, preds, sample_id, event_id):
    """
    Per-event tau tagging efficiency:
      For each signal (tau) event, check if ≥1 of its jets is correctly tagged.
      Returns the fraction of signal events that pass.
 
    Physics meaning: this is the true event selection efficiency — the
    probability that a H→ττ event survives the tau tagger selection.
 
    Uses event_id (saved per-process in npz) to group jets into events.
    Only signal jets (sample_id == 0) are considered.
    """
    sig_mask   = sample_id == 0          # tau process jets
    ev_ids_sig = event_id[sig_mask]      # event index for each signal jet
    preds_sig  = preds[sig_mask]
 
    if len(ev_ids_sig) == 0:
        return np.nan
 
    # For each unique event: did at least one of its jets get tagged?
    unique_events = np.unique(ev_ids_sig)
    n_events      = len(unique_events)
    n_tagged      = 0
 
    for ev in unique_events:
        jets_in_ev = preds_sig[ev_ids_sig == ev]
        if jets_in_ev.any():
            n_tagged += 1
 
    return n_tagged / n_events

In [4]:
def evaluate_model(model_key):
    cfg          = MODEL_CONFIGS[model_key]
    train_energy = cfg["train_energy"]
    results_dir  = cfg["results_dir"]
    label        = cfg["label"]
    algo         = cfg["type"]
    mnum         = "1" if train_energy == 125 else "2"
 
    print(f"\n{'='*60}")
    print(f"Evaluating {label}")
    print(f"{'='*60}")
 
    model_path = os.path.join(results_dir, cfg["model_file"])
    if not os.path.exists(model_path):
        print(f"  Model not found, skipping: {model_path}")
        return None
 
    model  = load_model(cfg)
    scaler = load_scaler(os.path.join(results_dir, cfg["scaler_file"]))
    print(f"  Loaded: {cfg['model_file']}")
 
    results = {}
 
    for energy in TEST_ENERGIES:
        dt       = np.load(os.path.join(FEATURE_DIR, f"bdt_features_{energy}GeV_test.npz"),
                           allow_pickle=True)
        X_test   = scaler.transform(dt["features"].astype(np.float32))
        y_test   = dt["labels"].astype(np.int32)
        sid      = dt["sample_id"].astype(np.int32)
        jet_pt   = dt["jet_pt"].astype(np.float32)
        event_id = dt["event_id"].astype(np.int64)
 
        scores = model.predict_proba(X_test)[:, 1]
 
        # ── standard metrics ───────────────────────────────────────────────
        auc    = roc_auc_score(y_test, scores)
        thresh = optimal_threshold(y_test, scores)
        preds  = (scores >= thresh).astype(int)
 
        acc  = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds, zero_division=0)
        rec  = recall_score(y_test, preds, zero_division=0)    # = jet-level sig eff
        f1   = f1_score(y_test, preds, zero_division=0)
 
        # ── per-event tau efficiency ────────────────────────────────────────
        # jet-level: rec is already this (TPR on tau jets)
        # event-level: ≥1 tagged jet per signal event
        evt_eff = per_event_tau_efficiency(y_test, preds, sid, event_id)
 
        # ── background rejection at working points ──────────────────────────
        brej = {wp: bkg_rejection_at_wp(y_test, scores, wp) for wp in WORKING_POINTS}
 
        # ── per-process AUC ────────────────────────────────────────────────
        m_jj   = (sid == 0) | (sid == 1)
        m_bb   = (sid == 0) | (sid == 2)
        auc_jj = roc_auc_score(y_test[m_jj], scores[m_jj]) if m_jj.sum() > 0 else np.nan
        auc_bb = roc_auc_score(y_test[m_bb], scores[m_bb]) if m_bb.sum() > 0 else np.nan
 
        # ── ROC curves ─────────────────────────────────────────────────────
        fpr,    tpr,    _ = roc_curve(y_test, scores)
        fpr_jj, tpr_jj, _ = roc_curve(y_test[m_jj], scores[m_jj])
        fpr_bb, tpr_bb, _ = roc_curve(y_test[m_bb], scores[m_bb])
 
        # ── pT-binned efficiency ────────────────────────────────────────────
        pt_centres, sig_eff_pt, bkg_fake_pt = pt_binned_efficiency(
            y_test, scores, jet_pt, thresh, PT_BINS
        )
 
        results[energy] = dict(
            auc=float(auc), auc_jj=float(auc_jj), auc_bb=float(auc_bb),
            accuracy=float(acc), precision=float(prec),
            recall=float(rec), f1=float(f1),
            jet_eff=float(rec),          # alias: rec IS the jet-level tau efficiency
            event_eff=float(evt_eff),
            brej=brej, threshold=float(thresh),
            fpr=fpr, tpr=tpr,
            fpr_jj=fpr_jj, tpr_jj=tpr_jj,
            fpr_bb=fpr_bb, tpr_bb=tpr_bb,
            pt_centres=pt_centres,
            sig_eff_pt=sig_eff_pt,
            bkg_fake_pt=bkg_fake_pt,
        )
 
        marker = " ◄ (in-dist.)" if energy == train_energy else ""
        print(f"\n  {energy} GeV{marker}")
        print(f"    AUC={auc:.4f}  (jj:{auc_jj:.4f}  bb:{auc_bb:.4f})")
        print(f"    Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  F1={f1:.4f}")
        print(f"    Jet-level tau eff  = {rec:.4f}")
        print(f"    Event-level tau eff= {evt_eff:.4f}")
        for wp in WORKING_POINTS:
            print(f"    BkgRej @ {int(wp*100)}%  = {brej[wp]:.1f}")
 
    # ── METRICS TABLE ──────────────────────────────────────────────────────
    # Columns: Energy | AUC | AUC_jj | AUC_bb | Acc | Prec | Rec | F1 |
    #          Jet-eff | Evt-eff | BRej@30% | BRej@50% | BRej@70% | BRej@90%
    lines = [
        f"{label} — Cross-Energy Metrics",
        "=" * 110,
        f"{'Energy':>8}  {'AUC':>6}  {'AUC_jj':>7}  {'AUC_bb':>7}  "
        f"{'Acc':>6}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}  "
        f"{'JetEff':>7}  {'EvtEff':>7}  "
        + "  ".join([f"BRej{int(w*100)}%" for w in WORKING_POINTS]),
        "-" * 110,
    ]
    for e in TEST_ENERGIES:
        r = results[e]
        brejs = "  ".join([f"{r['brej'][w]:>9.1f}" for w in WORKING_POINTS])
        m = " ◄" if e == train_energy else ""
        lines.append(
            f"{e:>7}GeV  {r['auc']:.4f}  {r['auc_jj']:.4f}   {r['auc_bb']:.4f}   "
            f"{r['accuracy']:.4f}  {r['precision']:.4f}  {r['recall']:.4f}  {r['f1']:.4f}  "
            f"{r['jet_eff']:>7.4f}  {r['event_eff']:>7.4f}  {brejs}{m}"
        )
    lines += [
        "=" * 110,
        "◄ = in-distribution (same energy as training)",
        "Jet-eff  : fraction of tau jets (in pT window) correctly tagged",
        "Evt-eff  : fraction of tau EVENTS with ≥1 jet correctly tagged (event selection efficiency)",
    ]
    table_str = "\n".join(lines)
    print("\n" + table_str)
 
    table_path = os.path.join(results_dir, f"{algo}_{model_key}_metrics_table.txt")
    with open(table_path, "w", encoding="utf-8") as f:
        f.write(table_str + "\n")
    print(f"\n  Saved: {os.path.basename(table_path)}")
 
    # ── SAVE METRICS NPZ ───────────────────────────────────────────────────
    np.savez_compressed(
        os.path.join(results_dir, f"{algo}_{model_key}_metrics.npz"),
        test_energies = np.array(TEST_ENERGIES),
        auc       = np.array([results[e]["auc"]        for e in TEST_ENERGIES]),
        auc_jj    = np.array([results[e]["auc_jj"]     for e in TEST_ENERGIES]),
        auc_bb    = np.array([results[e]["auc_bb"]     for e in TEST_ENERGIES]),
        accuracy  = np.array([results[e]["accuracy"]   for e in TEST_ENERGIES]),
        precision = np.array([results[e]["precision"]  for e in TEST_ENERGIES]),
        recall    = np.array([results[e]["recall"]     for e in TEST_ENERGIES]),
        f1        = np.array([results[e]["f1"]         for e in TEST_ENERGIES]),
        jet_eff   = np.array([results[e]["jet_eff"]    for e in TEST_ENERGIES]),
        event_eff = np.array([results[e]["event_eff"]  for e in TEST_ENERGIES]),
        brej_30   = np.array([results[e]["brej"][0.30] for e in TEST_ENERGIES]),
        brej_50   = np.array([results[e]["brej"][0.50] for e in TEST_ENERGIES]),
        brej_70   = np.array([results[e]["brej"][0.70] for e in TEST_ENERGIES]),
        brej_90   = np.array([results[e]["brej"][0.90] for e in TEST_ENERGIES]),
        train_energy = np.array([train_energy]),
    )
 
    # ═══════════════════════════════════════════════════════════════════════
    # PLOTS
    # ═══════════════════════════════════════════════════════════════════════
 
    energies  = np.array(TEST_ENERGIES)
    cmap      = plt.cm.plasma
    e_colors  = [cmap(i / (len(TEST_ENERGIES) - 1)) for i in range(len(TEST_ENERGIES))]
    wp_colors = ["royalblue", "darkorange", "forestgreen", "crimson"]
 
    # ── PLOT 1: Background rejection curve (1/FPR vs TPR), all energies ────
    fig, ax = plt.subplots(figsize=(7, 6))
    for i, energy in enumerate(TEST_ENERGIES):
        r     = results[energy]
        valid = r["fpr"] > 0
        brej  = np.where(r["fpr"] > 0, 1.0 / r["fpr"], np.nan)
        ls    = "--" if energy == train_energy else "-"
        lw    = 2.2  if energy == train_energy else 1.5
        lbl   = f"{energy} GeV  AUC={r['auc']:.4f}"
        if energy == train_energy: lbl += " (in-dist.)"
        ax.plot(r["tpr"][valid], brej[valid], color=e_colors[i], lw=lw, ls=ls, label=lbl)
    ax.set_yscale("log")
    ax.set_xlabel("Signal Efficiency (TPR)", fontsize=12)
    ax.set_ylabel("Background Rejection  (1 / FPR)", fontsize=12)
    ax.set_title(f"{label}\nBackground Rejection vs Signal Efficiency", fontsize=12)
    ax.legend(fontsize=8, loc="upper right")
    ax.set_xlim(0, 1); ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f"{algo}_{model_key}_bkg_rejection_curve.png"), dpi=150)
    plt.close()
 
    # ── PLOT 2: Background rejection at WPs vs energy ───────────────────────
    fig, ax = plt.subplots(figsize=(7, 5))
    for j, wp in enumerate(WORKING_POINTS):
        brej_arr  = np.array([results[e]["brej"][wp] for e in TEST_ENERGIES])
        safe_brej = np.where(np.isinf(brej_arr),
                             np.nanmax(brej_arr[~np.isinf(brej_arr)]) * 2, brej_arr)
        ax.plot(energies, safe_brej, "o-", color=wp_colors[j], lw=1.8, ms=6,
                label=f"BkgRej @ {int(wp*100)}% sig eff")
    ax.axvline(train_energy, color="gray", ls=":", lw=1.5, label=f"In-dist. ({train_energy} GeV)")
    ax.set_yscale("log")
    ax.set_xlabel("Test Energy  √s [GeV]", fontsize=12)
    ax.set_ylabel("Background Rejection  (1 / FPR)", fontsize=12)
    ax.set_title(f"{label}\nBackground Rejection at Working Points vs Energy", fontsize=12)
    ax.legend(fontsize=9); ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f"{algo}_{model_key}_brej_vs_energy.png"), dpi=150)
    plt.close()
 
    # ── PLOT 3: pT-binned tau efficiency — all 6 test energies ─────────────
    # (single-energy version removed — this all-energy version contains it)
    cmap2  = plt.cm.viridis
    cols2  = [cmap2(i / (len(TEST_ENERGIES) - 1)) for i in range(len(TEST_ENERGIES))]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for i, energy in enumerate(TEST_ENERGIES):
        r   = results[energy]
        ls  = "--" if energy == train_energy else "-"
        lw  = 2.0  if energy == train_energy else 1.3
        lbl = f"{energy} GeV" + (" (in-dist.)" if energy == train_energy else "")
        axes[0].plot(r["pt_centres"], r["sig_eff_pt"],  color=cols2[i], lw=lw, ls=ls,
                     marker="o", ms=4, label=lbl)
        axes[1].plot(r["pt_centres"], r["bkg_fake_pt"], color=cols2[i], lw=lw, ls=ls,
                     marker="s", ms=4, label=lbl)
 
    axes[0].set_xlabel("Jet $p_T$  [GeV]", fontsize=12)
    axes[0].set_ylabel("τ Tagging Efficiency (jet-level)", fontsize=12)
    axes[0].set_title("τ Tagging Efficiency vs $p_T$ — All Test Energies", fontsize=11)
    axes[0].set_ylim(0, 1.05); axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=8, loc="lower right")
 
    axes[1].set_xlabel("Jet $p_T$  [GeV]", fontsize=12)
    axes[1].set_ylabel("Background Fake Rate", fontsize=12)
    axes[1].set_title("Background Fake Rate vs $p_T$ — All Test Energies", fontsize=11)
    axes[1].grid(True, alpha=0.3); axes[1].legend(fontsize=8)
 
    fig.suptitle(label, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f"{algo}_{model_key}_pt_efficiency_all_energies.png"), dpi=150)
    plt.close()
 
    # ── PLOT 4: Per-event tau efficiency vs energy ──────────────────────────
    # Physics plot: shows what fraction of H→ττ events survive the tagger
    # at each test energy. This is the event-selection efficiency.
    jet_effs = np.array([results[e]["jet_eff"]   for e in TEST_ENERGIES])
    evt_effs = np.array([results[e]["event_eff"] for e in TEST_ENERGIES])
 
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(energies, jet_effs, "o-",  color="royalblue",  lw=2, ms=7,
            label="Jet-level τ efficiency")
    ax.plot(energies, evt_effs, "s--", color="darkorange", lw=2, ms=7,
            label="Event-level τ efficiency (≥1 jet tagged)")
    ax.axvline(train_energy, color="gray", ls=":", lw=1.5,
               label=f"In-dist. ({train_energy} GeV)")
    ax.set_xlabel("Test Energy  √s [GeV]", fontsize=12)
    ax.set_ylabel("τ Tagging Efficiency", fontsize=12)
    ax.set_title(f"{label}\nτ Tagging Efficiency vs Energy\n"
                 f"(jet-level vs event-level)", fontsize=11)
    ax.set_ylim(max(0, min(jet_effs.min(), evt_effs.min()) - 0.02), 1.01)
    ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f"{algo}_{model_key}_tau_efficiency_vs_energy.png"), dpi=150)
    plt.close()
 
    # ── PLOT 5: Per-process rejection curves — all energies ─────────────────
    fig, ax = plt.subplots(figsize=(7, 6))
    from matplotlib.lines import Line2D
    for i, energy in enumerate(TEST_ENERGIES):
        r        = results[energy]
        ls       = "--" if energy == train_energy else "-"
        valid_jj = r["fpr_jj"] > 0
        valid_bb = r["fpr_bb"] > 0
        brej_jj  = np.where(r["fpr_jj"] > 0, 1.0 / r["fpr_jj"], np.nan)
        brej_bb  = np.where(r["fpr_bb"] > 0, 1.0 / r["fpr_bb"], np.nan)
        ax.plot(r["tpr_jj"][valid_jj], brej_jj[valid_jj],
                color=e_colors[i], lw=1.5, ls=ls, alpha=0.85)
        ax.plot(r["tpr_bb"][valid_bb], brej_bb[valid_bb],
                color=e_colors[i], lw=1.5, ls=":", alpha=0.85)
    ax.set_yscale("log")
    ax.set_xlabel("τ Signal Efficiency", fontsize=12)
    ax.set_ylabel("Background Rejection  (1 / FPR)", fontsize=12)
    ax.set_title(f"{label}\nPer-Process Background Rejection\n(solid=τ vs jj, dotted=τ vs bb)", fontsize=11)
    legend_els = [
        Line2D([0], [0], color="gray", lw=1.5, ls="-",  label="τ vs jj"),
        Line2D([0], [0], color="gray", lw=1.5, ls=":",  label="τ vs bb"),
        Line2D([0], [0], color="gray", lw=2.0, ls="--", label="In-distribution energy"),
    ]
    ax.legend(handles=legend_els, fontsize=9); ax.set_xlim(0, 1)
    ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f"{algo}_{model_key}_perprocess_rejection.png"), dpi=150)
    plt.close()
 
    # ── PLOT 6: AUC vs energy ─────────────────────────────────────────────
    aucs    = np.array([results[e]["auc"]    for e in TEST_ENERGIES])
    aucs_jj = np.array([results[e]["auc_jj"] for e in TEST_ENERGIES])
    aucs_bb = np.array([results[e]["auc_bb"] for e in TEST_ENERGIES])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(energies, aucs,    "o-",  color="royalblue",  lw=2,   ms=7, label="Overall AUC")
    ax.plot(energies, aucs_jj, "s--", color="darkorange", lw=1.5, ms=6, label="AUC (τ vs jj)")
    ax.plot(energies, aucs_bb, "^--", color="forestgreen",lw=1.5, ms=6, label="AUC (τ vs bb)")
    ax.axvline(train_energy, color="gray", ls=":", lw=1.5, label=f"In-dist. ({train_energy} GeV)")
    ax.set_xlabel("Test Energy  √s [GeV]", fontsize=12)
    ax.set_ylabel("AUC", fontsize=12)
    ax.set_title(f"{label} — AUC vs Test Energy", fontsize=12)
    ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
    ax.set_ylim(min(aucs_jj.min(), aucs_bb.min()) - 0.002, 1.001)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f"{algo}_{model_key}_auc_vs_energy.png"), dpi=150)
    plt.close()
 
    # ── PLOT 7: Feature importance ────────────────────────────────────────
    try:
        imp        = model.feature_importances_
        sorted_idx = np.argsort(imp)
        fig, ax = plt.subplots(figsize=(7, 5))
        bars = ax.barh([FEATURE_NAMES[i] for i in sorted_idx], imp[sorted_idx],
                       color="steelblue", alpha=0.85)
        ax.set_xlabel("Feature Importance (normalised gain)", fontsize=12)
        ax.set_title(f"{label}\nFeature Importance", fontsize=12)
        ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=3)
        ax.set_xlim(0, imp.max() * 1.15); ax.grid(True, axis="x", alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(results_dir, f"{algo}_{model_key}_feature_importance.png"), dpi=150)
        plt.close()
    except Exception as e:
        print(f"  Feature importance skipped: {e}")
 
    print(f"\n  All plots saved → {results_dir}")
    return results

In [5]:
all_results = {}
for key, cfg in MODEL_CONFIGS.items():
    model_path = os.path.join(cfg["results_dir"], cfg["model_file"])
    if os.path.exists(model_path):
        all_results[key] = evaluate_model(key)
    else:
        print(f"\nSkipping {key} — not found: {model_path}")
 
print(f"\n{'='*60}")
print(f"Evaluation complete. {len(all_results)} model(s) evaluated.")
print("="*60)


Evaluating Random Forest (125 GeV)
  Loaded: rf_model1_125GeV.pkl


[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.2s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.8s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    1.8s
[Parallel(n_jobs=10)]: Done 700 out of 700 | elapsed:    2.8s finished



  100 GeV
    AUC=0.9979  (jj:0.9963  bb:0.9993)
    Acc=0.9849  Prec=0.9633  Rec=0.9640  F1=0.9636
    Jet-level tau eff  = 0.9640
    Event-level tau eff= 0.9756
    BkgRej @ 30%  = 3229.9
    BkgRej @ 50%  = 1615.0
    BkgRej @ 70%  = 782.6
    BkgRej @ 90%  = 275.0


[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.1s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.6s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    1.4s
[Parallel(n_jobs=10)]: Done 700 out of 700 | elapsed:    2.3s finished



  125 GeV ◄ (in-dist.)
    AUC=0.9986  (jj:0.9977  bb:0.9995)
    Acc=0.9879  Prec=0.9683  Rec=0.9752  F1=0.9717
    Jet-level tau eff  = 0.9752
    Event-level tau eff= 0.9841
    BkgRej @ 30%  = 4391.8
    BkgRej @ 50%  = 2303.0
    BkgRej @ 70%  = 1259.0
    BkgRej @ 90%  = 442.3


...


  All plots saved → E:/Python/MSc_Project_Upgrade/results_analysis/bdt\XGB_model2_250GeV

Evaluating LightGBM (125 GeV)
  Loaded: lgbm_model1_125GeV.txt

  100 GeV
    AUC=0.9981  (jj:0.9967  bb:0.9995)
    Acc=0.9852  Prec=0.9603  Rec=0.9687  F1=0.9645
    Jet-level tau eff  = 0.9687
    Event-level tau eff= 0.9785
    BkgRej @ 30%  = 4555.0
    BkgRej @ 50%  = 1910.2
    BkgRej @ 70%  = 906.4
    BkgRej @ 90%  = 292.7

  125 GeV ◄ (in-dist.)
    AUC=0.9987  (jj:0.9979  bb:0.9996)
    Acc=0.9882  Prec=0.9665  Rec=0.9783  F1=0.9724
    Jet-level tau eff  = 0.9783
    Event-level tau eff= 0.9857
    BkgRej @ 30%  = 5901.4
    BkgRej @ 50%  = 2861.3
    BkgRej @ 70%  = 1441.6
    BkgRej @ 90%  = 447.5

  150 GeV
    AUC=0.9991  (jj:0.9984  bb:0.9997)
    Acc=0.9903  Prec=0.9772  Rec=0.9772  F1=0.9772
    Jet-level tau eff  = 0.9772
    Event-level tau eff= 0.9861
    BkgRej @ 30%  = 10172.0
    BkgRej @ 50%  = 3281.3
    BkgRej @ 70%  = 1919.2
    BkgRej @ 90%  = 637.7

  200 GeV
    AU

C:\Users\User\AppData\Local\Temp\ipykernel_1316\194977128.py:162: RuntimeWarning: divide by zero encountered in divide
  brej  = np.where(r["fpr"] > 0, 1.0 / r["fpr"], np.nan)
C:\Users\User\AppData\Local\Temp\ipykernel_1316\194977128.py:258: RuntimeWarning: divide by zero encountered in divide
  brej_jj  = np.where(r["fpr_jj"] > 0, 1.0 / r["fpr_jj"], np.nan)
C:\Users\User\AppData\Local\Temp\ipykernel_1316\194977128.py:259: RuntimeWarning: divide by zero encountered in divide
  brej_bb  = np.where(r["fpr_bb"] > 0, 1.0 / r["fpr_bb"], np.nan)



  All plots saved → E:/Python/MSc_Project_Upgrade/results_analysis/bdt\LGBM_model1_125GeV

Evaluating LightGBM (250 GeV)
  Loaded: lgbm_model2_250GeV.txt

  100 GeV
    AUC=0.9974  (jj:0.9955  bb:0.9992)
    Acc=0.9821  Prec=0.9538  Rec=0.9602  F1=0.9570
    Jet-level tau eff  = 0.9602
    Event-level tau eff= 0.9710
    BkgRej @ 30%  = 3116.6
    BkgRej @ 50%  = 1432.6
    BkgRej @ 70%  = 759.2
    BkgRej @ 90%  = 211.0

  125 GeV
    AUC=0.9984  (jj:0.9973  bb:0.9995)
    Acc=0.9862  Prec=0.9672  Rec=0.9675  F1=0.9674
    Jet-level tau eff  = 0.9675
    Event-level tau eff= 0.9770
    BkgRej @ 30%  = 3631.7
    BkgRej @ 50%  = 2303.0
    BkgRej @ 70%  = 1267.4
    BkgRej @ 90%  = 363.2

  150 GeV
    AUC=0.9989  (jj:0.9981  bb:0.9997)
    Acc=0.9893  Prec=0.9712  Rec=0.9787  F1=0.9749
    Jet-level tau eff  = 0.9787
    Event-level tau eff= 0.9856
    BkgRej @ 30%  = 6781.3
    BkgRej @ 50%  = 2906.3
    BkgRej @ 70%  = 1681.3
    BkgRej @ 90%  = 552.8

  200 GeV
    AUC=0.9994  (jj

C:\Users\User\AppData\Local\Temp\ipykernel_1316\194977128.py:162: RuntimeWarning: divide by zero encountered in divide
  brej  = np.where(r["fpr"] > 0, 1.0 / r["fpr"], np.nan)
C:\Users\User\AppData\Local\Temp\ipykernel_1316\194977128.py:258: RuntimeWarning: divide by zero encountered in divide
  brej_jj  = np.where(r["fpr_jj"] > 0, 1.0 / r["fpr_jj"], np.nan)
C:\Users\User\AppData\Local\Temp\ipykernel_1316\194977128.py:259: RuntimeWarning: divide by zero encountered in divide
  brej_bb  = np.where(r["fpr_bb"] > 0, 1.0 / r["fpr_bb"], np.nan)



  All plots saved → E:/Python/MSc_Project_Upgrade/results_analysis/bdt\LGBM_model2_250GeV

Evaluation complete. 6 model(s) evaluated.
